In [ ]:
import os
from glob import glob
import geopandas
import pandas
import subprocess

In [ ]:
processed_data_path = 'L:\Jamaica\Inputs'
base_path = 'Z:\\jamaica\\Inputs'
output_path = 'Z:\\jamaica\\Results'

In [ ]:
network_csv = os.path.join(processed_data_path,
                            "networks",
                            "network_layers_hazard_intersections_details.csv")
hazard_csv = os.path.join(processed_data_path,
                            "coastal_flood_rasters.csv")
damage_curves_csv = os.path.join(processed_data_path,
                            "damage_curves",
                            "asset_damage_curve_mapping.csv")
hazard_damage_parameters_csv = os.path.join(processed_data_path,
                            "damage_curves",
                            "hazard_damage_parameters.csv")
damage_results_folder = "direct_damages"

In [ ]:
# Create a path to store intersection outputs
#output_path = os.path.join(output_path,"coastal_flood_intersections")
#if os.path.exists(output_path) == False:
    #os.mkdir(output_path)

In [ ]:
# vector_details_csv = os.path.join(base_path,"infrastructure","network_layers.csv")
# raster_details_csv = os.path.join(base_path,"coastal_floods_FN","hazard_layers.csv")

In [ ]:
#run_intersections = True  # Set to True is you want to run this process
#if run_intersections is True:
    #args = [
#             "python",
#             "vector_raster_intersections.py",
#             f"{vector_details_csv}",
#             f"{raster_details_csv}",
#             f"{output_path}"
#             ]
#     print ("* Start the processing of vector-raster intersections")
#     print (args)
#     subprocess.run(args)
# print ("* Done with the processing of vector-raster intersections")

In [ ]:
file_name = os.path.join(output_path, "airport_polygon_splits__hazard_layers_FN__areas.geoparquet")
df = geopandas.read_parquet(file_name)
df

In [ ]:
df.to_file(os.path.join(output_path, "airport_polygon_splits__hazard_layers_FN__areas.gpkg"), layer="area", driver="GPKG")

In [ ]:
"""Next we call the summary scripts
"""
args = [
        "python",
        "damage_calculations.py",
        f"{damage_results_folder}",
        f"{network_csv}",
        f"{hazard_csv}",
        f"{damage_curves_csv}",
        f"{hazard_damage_parameters_csv}",
        "0","0","0"
        ]
print ("* Start the processing of summarising damage results")
print (args)
subprocess.check_output(args)

In [ ]:
damage_results_folder = os.path.join(output_path, "direct_damages")
mangrove_flood_damage_columns = ["coastal_flood_fn_mg_rp_25",
                       "coastal_flood_fn_mg_rp_100",
                       "coastal_flood_fn_mg_rp_500"]
nomangrove_flood_damage_columns = ["coastal_flood_fn_nomg_rp_25",
                       "coastal_flood_fn_nomg_rp_100",
                       "coastal_flood_fn_nomg_rp_500"]
difference_columns = ["coastal_flood_diff_rp_25",
                       "coastal_flood_diff_rp_100",
                       "coastal_flood_diff_rp_500"]
flood_damage_columns = mangrove_flood_damage_columns + nomangrove_flood_damage_columns + difference_columns 

In [ ]:
jamaica_crs = 3448

asset_data_details = pandas.read_csv(network_csv)
damage_totals = [] # List object to assemble many dataframes 
for asset_info in asset_data_details.itertuples():
        asset_path = asset_info.path
        asset_gpkg = asset_info.asset_gpkg
        asset_layer = asset_info.asset_layer
        asset_id = asset_info.asset_id_column
        df_path = os.path.join(output_path,"direct_damages",
                               f"{asset_gpkg}_{asset_layer}",
                               f"{asset_gpkg}_{asset_layer}_direct_damages_parameter_set_0.parquet")
        if os.path.exists(df_path):
            df = pandas.read_parquet(df_path) #read in the files using the .parquet from Raghav's code
            df[difference_columns] = df[nomangrove_flood_damage_columns] - df[mangrove_flood_damage_columns].values
            df = df.groupby([asset_id]).sum(flood_damage_columns).reset_index() # calculate asset level damages = .groupby(node_id).sum()
            df_path = os.path.join(processed_data_path,
                                   f"{asset_path}")
            df_geom = geopandas.read_file(df_path, layer = asset_layer)
            df_geom = df_geom.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
            df = pandas.merge(df,df_geom[[asset_id,"geometry"]],how="left",on=[asset_id])
            df = geopandas.GeoDataFrame(df,geometry="geometry",crs=jamaica_crs)
            print(df)
            df.to_file(os.path.join(output_path,
                                    'damage_estimates', 
                                    f'{asset_gpkg}_{asset_layer}_asset_damages_groupedby.gpkg'),
                                    driver='GPKG')
            df['sector'] = asset_gpkg
            df['layer'] = asset_layer
            df = df.groupby(['sector','layer']).sum(flood_damage_columns).reset_index()
            damage_totals.append(df) # Add things to list

# Convert list of dataframes to 1 dataframe by concatenation
damage_totals = pandas.concat(damage_totals,axis=0,ignore_index=True)
damage_totals.to_csv(os.path.join(output_path,
                                    'damage_estimates', 
                                    'asset_damages_groupedby.csv'))

In [ ]:
print(asset_gpkg)

In [ ]:


            # Calculate total sector damages with and without mangroves for each return period

# total_sector_damages_mg_rp_25 = sum(coastal_flood_fn_mg_rp_25)

In [ ]:
# buffer the mangroves by 1km; intersect with all the assets and output a list of mangrove ID and asset ID OR do nearest neighbour

jamaica_crs = 3448

FN_Mangroves = geopandas.read_file(os.path.join(processed_data_path,"Forces of nature mangroves", 'mangroves.shp'))[["ID","geometry"]]
FN_Mangroves = FN_Mangroves.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system

#FN_Mangroves["geometry"] = FN_Mangroves.buffer(distance=1000)

asset_data_details = pandas.read_csv(network_csv)
for asset_info in asset_data_details.itertuples():
        asset_path = asset_info.path
        asset_gpkg = asset_info.asset_gpkg
        asset_layer = asset_info.asset_layer
        asset_id = asset_info.asset_id_column
        df_path = os.path.join(output_path,"direct_damages",
                               f"{asset_gpkg}_{asset_layer}",
                               f"{asset_gpkg}_{asset_layer}_direct_damages_parameter_set_0.parquet")
        if os.path.exists(df_path):
            df = pandas.read_parquet(df_path)
            df[difference_columns] = df[nomangrove_flood_damage_columns] - df[mangrove_flood_damage_columns].values
            df = df.groupby([asset_id]).sum(flood_damage_columns).reset_index()
            df_path = os.path.join(processed_data_path,
                                   f"{asset_path}")
            df_geom = geopandas.read_file(df_path, layer = asset_layer)
            df_geom = df_geom.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
            df = pandas.merge(df,df_geom[[asset_id,"geometry"]],how="left",on=[asset_id])
            df = geopandas.GeoDataFrame(df,geometry="geometry",crs=jamaica_crs)
            #mangroves_asset_intersection = FN_Mangroves[["ID","geometry"]] \
            #.overlay(df,how="intersection",keep_geom_type=False)
            #print(mangroves_asset_intersection[["ID",asset_id]])
        
        #for loop to calculate all the mangrove ID distances from an asset ID
        
        for asset_info in asset_data_details.itertuples():
            asset_path = asset_info.path
            asset_gpkg = asset_info.asset_gpkg
            asset_layer = asset_info.asset_layer
            asset_id = asset_info.asset_id_column
            df_path = os.path.join(output_path,"direct_damages",
                               f"{asset_gpkg}_{asset_layer}",
                               f"{asset_gpkg}_{asset_layer}_direct_damages_parameter_set_0.parquet")
        

In [ ]:
asset_data_details = pandas.read_csv(network_csv)
mapping_result = []
for asset_info in asset_data_details.itertuples():
        asset_path = asset_info.path
        asset_layer = asset_info.asset_layer
        asset_id = asset_info.asset_id_column
        df_path = os.path.join(processed_data_path,
                               f"{asset_path}")
        df = geopandas.read_file(df_path, layer = asset_layer)
        df = df.to_crs(epsg=jamaica_crs) # Convert geometry to Jamaica projection system
        #mangroves_asset_intersection = FN_Mangroves[["ID","geometry"]] \
        #.overlay(df,how="intersection",keep_geom_type=False)
        #print(mangroves_asset_intersection[["ID",asset_id]])
        #intersect buffered mangrove with the output of the above loop - with the asset id 
        


In [ ]:
# direct_damages_file = os.path.join(hazard_asset_intersection_path,
  #                              f"{asset_info.asset_gpkg}_with_coastal_{asset_info.asset_layer}.parquet")
   #     if os.path.isfile(hazard_intersection_file) is True:
    #        hazard_df = geopandas.read_parquet(hazard_intersection_file)

In [ ]:
#run_intersections = True  # Set to True is you want to run this process
#if run_intersections is True:
    #args = [
#             "python",
#             "vector_raster_intersections.py",
#             f"{vector_details_csv}",
#             f"{raster_details_csv}",
#             f"{output_path}"
#             ]
#     print ("* Start the processing of vector-raster intersections")
#     print (args)
#     subprocess.run(args)
# print ("* Done with the processing of vector-raster intersections")

In [ ]:
# df.to_file(os.path.join(output_path, "airport_polygon_splits__hazard_layers_FN__areas.gpkg"), layer="area", driver="GPKG")